##### 13 — Explainable AI for Images



##### 1. Objective

This notebook applies Grad-CAM to the fine-tuned ResNet18 product-defect
classifier to investigate which broad spatial regions positively influence
individual class scores.

The notebook will:

1. Load the persisted fine-tuned ResNet18 model from MLflow.
2. Recreate the deterministic pretrained evaluation pipeline.
3. Generate Grad-CAM explanations using the final convolutional stage.
4. Inspect a correctly classified defect image.
5. Analyze known false-negative defect predictions.
6. Compare explanations for the predicted and actual classes.
7. Compare Layer3 and Layer4 Grad-CAM explanations.
8. Document the capabilities and limitations of Grad-CAM.

> 🔍 **Explainable AI (XAI):**
> Grad-CAM helps investigate where positive influence for a selected class
> is spatially concentrated.

> ⚖️ **Responsible AI (RAI):**
> Grad-CAM is not ground-truth defect localization and should not be
> presented as proof that a highlighted region contains the physical defect.

**2. Imports**

In [0]:
import os

import matplotlib.pyplot as plt
import mlflow
import torch
from PIL import Image

from src.data_preparation import (
    create_image_dataloaders,
)
from src.explainability import GradCAM

**3. Configuration**

In [0]:
RESNET18_RUN_ID = "3f9b44ae55a449c5850f0b48c4ee4fa6"


RESNET18_MODEL_URI = f"runs:/{RESNET18_RUN_ID}/model"


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Model URI:", RESNET18_MODEL_URI)

**4. Load the persisted ResNet18 model**

In [0]:
model = mlflow.pytorch.load_model(RESNET18_MODEL_URI)

model = model.to(device)
model.eval()

print(model)

Expected classifier: Linear(in_features=512, out_features=2, bias=True)

Concept:

The model is loaded from the persisted MLflow artifact rather than retrained.

The fine-tuned ResNet18 produces two output logits:

- `def_front`
- `ok_front`

Grad-CAM will explain one selected class score at a time.

**5. Recreate the deterministic evaluation pipeline**

In [0]:
data = create_image_dataloaders(
    preprocessing_mode="pretrained"
)

print("Classes:", data.class_names)
print("Class mapping:", data.class_to_idx)
print("Test images:", len(data.test_dataset))

Grad-CAM is class-specific.

We don't merely ask: What did the network see?

We ask something closer to: 

- Which spatial features influenced the score for class 0? or:
- Which spatial features influenced the score for class 1?


Why evaluation preprocessing?

- Grad-CAM should explain the same deterministic preprocessing pipeline used during model evaluation.
- Random training augmentation is therefore not used when generating these explanations.

**6. Select the primary Grad-CAM target layer**

In [0]:
target_layer = model.layer4[-1].conv2

print(target_layer)

You'll see something structurally similar to:

Sequential(
  (0): BasicBlock(...)
  (1): BasicBlock(...)
)

This is worth stopping at before we write the Grad-CAM implementation.

Remember:

``` text

layer4
   │
   ├── BasicBlock 0
   │
   └── BasicBlock 1

```

So layer4 itself isn't one convolution.

Inside the final BasicBlock, there are convolutional operations.


Conceptually:

``` text

model
  ↓
layer4
  ↓
last BasicBlock
  ↓
conv2
  ↑
Grad-CAM target

```

Why this layer?
`model.layer4[-1].conv2` is a late convolutional layer in ResNet18.

For a 224 × 224 model input, its output is approximately:     [batch, 512, 7, 7]

This provides:

- 512 learned feature channels
- a 7 × 7 spatial representation

Late convolutional features are relatively close to the final
classification decision while still retaining spatial information.

The 7 × 7 spatial resolution is coarse, so Grad-CAM explanations from this
layer should be interpreted as broad influential regions rather than precise
defect boundaries.

**7. Grad-CAM calculation**

**Grad-CAM Process**

For one image:

    Image
    [1, 3, 224, 224]

            ↓ forward pass

    Target convolution

            ↓

    512 activation grids
    each 7 × 7

            ↓

    Selected class logit

            ↓ backward pass

    512 gradient grids
    each 7 × 7

            ↓

    Average each gradient grid
    across height and width

            ↓

    512 channel-importance values

            ↓

    importance₁ × activation grid₁
    +
    importance₂ × activation grid₂
    +
    ...
    +
    importance₅₁₂ × activation grid₅₁₂

            ↓

    one 7 × 7 Grad-CAM grid

            ↓ ReLU
            ↓ normalize
            ↓ interpolate

    224 × 224 [heatmap](url)

**Important distinction:**

- **Activations** come from the normal forward pass.
- **Gradients** come from the backward pass for the selected class.
- Gradients are averaged to estimate channel importance.
- Channel importance values weight the corresponding activation grids.
- The weighted activation grids are combined into the Grad-CAM heatmap.

The Grad-CAM backward pass does not train the model because no optimizer
update is performed.

Earlier I said Grad-CAM commonly targets something such as: model.layer4

That's conceptually fine when discussing the final convolutional block.

For our implementation, however, I recommend being more precise:

target_layer = (
    model.layer4[-1].conv2
)

This gives us a specific convolutional layer from which to capture activations and gradients.

** What shape should we expect?**

For a normal ResNet18 224×224 input, the late feature representation is spatially much smaller than the original image.

Conceptually:

Input

[1, 3, 224, 224]

        ↓

late convolutional features

approximately
[1, 512, 7, 7]

Interpretation:

1
→ one image

512
→ 512 learned feature maps

7 × 7
→ spatial locations

That explains something you asked earlier about Grad-CAM being broad or fuzzy.

We're not generating the heatmap directly at:

224 × 224

We're starting with a much coarser representation such as:

7 × 7

and later enlarging it back toward the original image size.

So:

7 × 7 Grad-CAM information
          ↓
upsampling
          ↓
224 × 224 visualization

That's one reason Grad-CAM should not be interpreted like precise segmentation.

**Understanding layer4**

output shows:

``` text

layer4
│
├── BasicBlock (0)
│   ├── conv1: 256 → 512
│   ├── bn1
│   ├── relu
│   ├── conv2: 512 → 512
│   ├── bn2
│   └── downsample: 256 → 512
│
└── BasicBlock (1)
    ├── conv1: 512 → 512
    ├── bn1
    ├── relu
    ├── conv2: 512 → 512    ← our target
    └── bn2
    
```

Notice something important in block 0:

conv1:
Conv2d(
    256,
    512,
    kernel_size=3,
    stride=2,
)

The stride=2 reduces the spatial dimensions while increasing channels:

approximately

[B, 256, 14, 14]
        ↓
[B, 512, 7, 7]

The second block keeps:

[B, 512, 7, 7]

because its convolutions use:

stride = 1
padding = 1
kernel = 3

So our target:

model.layer4[-1].conv2

produces approximately:

[B, 512, 7, 7]

That's exactly the type of representation Grad-CAM needs.

##### 8. Verify Grad-CAM on one image

In [0]:
sample_index = 0

image_tensor, actual_label = (
    data.test_dataset[sample_index]
)

image_batch = image_tensor.unsqueeze(0)

image_path, path_label = (
    data.test_dataset.samples[sample_index]
)

print("Image:", os.path.basename(image_path))
print("Image tensor:", image_tensor.shape)
print("Batch:", image_batch.shape)
print(
    "Actual class:",
    data.class_names[actual_label],
)

In [0]:
#Create Grad-CAM:

grad_cam = GradCAM(
    model=model,
    target_layer=target_layer,
)

result = grad_cam.generate(
    input_tensor=image_batch,
)

In [0]:
print(
    "Activation shape:",
    result.activation_shape,
)

print(
    "Gradient shape:",
    result.gradient_shape,
)

print(
    "Heatmap shape:",
    result.heatmap.shape,
)

print(
    "Actual class:",
    data.class_names[actual_label],
)

print(
    "Predicted class:",
    data.class_names[
        result.predicted_class
    ],
)

print(
    "Explained class:",
    data.class_names[
        result.target_class
    ],
)

print(
    "Probabilities:",
    result.probabilities,
)

**Interpretation**

The matching activation and gradient shapes confirm that Grad-CAM captured
one gradient value for every activation value at the target layer.

The 7 × 7 gradient grid for each channel is averaged to produce one
class-specific importance value for that channel.

The resulting 512 weighted activation grids are combined into one spatial
Grad-CAM grid.

##### 9. Visualize a correct prediction

In [0]:
original_image = Image.open(
    image_path
).convert("RGB")

display_image = original_image.resize(
    (224, 224)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
)

axes[0].imshow(display_image)
axes[0].set_title("Original Image")
axes[0].axis("off")


axes[1].imshow(
    result.heatmap.numpy(),
    cmap="jet",
)
axes[1].set_title(
    "Grad-CAM Heatmap"
)
axes[1].axis("off")


axes[2].imshow(display_image)
axes[2].imshow(
    result.heatmap.numpy(),
    cmap="jet",
    alpha=0.40,
)
axes[2].set_title(
    "Grad-CAM Overlay"
)
axes[2].axis("off")


plt.suptitle(
    f"Actual: "
    f"{data.class_names[actual_label]} | "
    f"Predicted: "
    f"{data.class_names[result.predicted_class]} | "
    f"Explained: "
    f"{data.class_names[result.target_class]}"
)

plt.tight_layout()
plt.show()

**🔍 XAI Observation**

The Grad-CAM overlay identifies broad spatial regions that positively
influenced the selected class score.

A stronger heatmap region does **not** mean that the corresponding pixels
are physically defective or non-defective.

The meaning of the heatmap always depends on the class being explained:

- when explaining `def_front`, stronger regions positively influence the
  defect score;
- when explaining `ok_front`, stronger regions positively influence the
  OK score.

Because the dataset contains image-level labels but no ground-truth defect
bounding boxes or segmentation masks, the heatmap cannot establish the true
physical defect location.

##### 10. Analyze known false negatives

In [0]:
#From the previous evaluation, ResNet18 produced three known defect false negatives.

FALSE_NEGATIVE_FILENAMES = [
    "cast_def_0_150.jpeg",
    "cast_def_0_1591.jpeg",
    "cast_def_0_2102.jpeg",
]

For each image we will generate two explanations:

Explain predicted class:
"What positively supported OK?"

Explain defect class:
"What positively influenced the DEFECT score?"

Generate the explanations:

In [0]:
false_negative_explanations = []

defect_class_index = (
    data.class_to_idx["def_front"]
)

for filename in FALSE_NEGATIVE_FILENAMES:

    matching_indices = [
        index
        for index, (path, label)
        in enumerate(data.test_dataset.samples)
        if path.endswith(filename)
    ]

    if len(matching_indices) != 1:
        raise ValueError(
            f"Expected exactly one match for "
            f"{filename}, found "
            f"{len(matching_indices)}."
        )

    sample_index = matching_indices[0]

    image_path, _ = (
        data.test_dataset.samples[
            sample_index
        ]
    )

    image_tensor, actual_label = (
        data.test_dataset[
            sample_index
        ]
    )

    image_batch = image_tensor.unsqueeze(0)

    predicted_explanation = (
        grad_cam.generate(
            input_tensor=image_batch,
        )
    )

    defect_explanation = (
        grad_cam.generate(
            input_tensor=image_batch,
            target_class=defect_class_index,
        )
    )

    false_negative_explanations.append(
        {
            "filename": filename,
            "image_path": image_path,
            "actual_label": actual_label,
            "predicted_explanation": (
                predicted_explanation
            ),
            "defect_explanation": (
                defect_explanation
            ),
        }
    )

##### 11. Verify false-negative predictions

In [0]:
ok_class_index = (
    data.class_to_idx["ok_front"]
)

for case in false_negative_explanations:

    explanation = (
        case["predicted_explanation"]
    )

    probabilities = (
        explanation.probabilities
    )

    print(
        "\n",
        case["filename"],
    )

    print(
        "Actual:",
        data.class_names[
            case["actual_label"]
        ],
    )

    print(
        "Predicted:",
        data.class_names[
            explanation.predicted_class
        ],
    )

    print(
        "def_front probability:",
        f"{probabilities[defect_class_index]:.2%}",
    )

    print(
        "ok_front probability:",
        f"{probabilities[ok_class_index]:.2%}",
    )

These cases demonstrate that incorrect predictions can occur at different
confidence levels.

The highest-confidence failure is especially important because:

High confidence does not imply correctness.

Confidence also does not explain why a prediction occurred.

##### 12. Compare false-negative Grad-CAM explanations

In [0]:
num_cases = len(
    false_negative_explanations
)

fig, axes = plt.subplots(
    num_cases,
    3,
    figsize=(15, 5 * num_cases),
)

for row, case in enumerate(
    false_negative_explanations
):

    original_image = Image.open(
        case["image_path"]
    ).convert("RGB")

    display_image = original_image.resize(
        (224, 224)
    )

    predicted_explanation = (
        case["predicted_explanation"]
    )

    defect_explanation = (
        case["defect_explanation"]
    )

    probabilities = (
        predicted_explanation.probabilities
    )

    ok_probability = probabilities[
        ok_class_index
    ].item()

    # Original image
    axes[row, 0].imshow(
        display_image
    )

    axes[row, 0].set_title(
        f"{case['filename']}\n"
        f"Actual: def_front | "
        f"Predicted: ok_front\n"
        f"OK probability: "
        f"{ok_probability:.2%}"
    )

    axes[row, 0].axis("off")

    # Explain predicted OK class
    axes[row, 1].imshow(
        display_image
    )

    axes[row, 1].imshow(
        predicted_explanation
        .heatmap
        .numpy(),
        cmap="jet",
        alpha=0.40,
    )

    axes[row, 1].set_title(
        "Grad-CAM: Why OK?"
    )

    axes[row, 1].axis("off")

    # Explain defect class
    axes[row, 2].imshow(
        display_image
    )

    axes[row, 2].imshow(
        defect_explanation
        .heatmap
        .numpy(),
        cmap="jet",
        alpha=0.40,
    )

    axes[row, 2].set_title(
        "Grad-CAM: Evidence for DEFECT?"
    )

    axes[row, 2].axis("off")


plt.suptitle(
    "ResNet18 False-Negative "
    "Grad-CAM Comparison",
    fontsize=16,
)

plt.tight_layout()
plt.show()

##### 13. False-negative findings

##### 🔍 XAI Findings — False Negatives

All three analyzed images have:

- actual class: `def_front`
- predicted class: `ok_front`

The `ok_front` Grad-CAM explanations show substantial positive influence
in broadly upper portions of the casting, although the exact highlighted
regions differ between images.

The `def_front` explanations are less spatially consistent. Two cases show
strong positive influence near the right-side/background region, while
another emphasizes a lower portion of the casting.

These observations identify patterns worth investigating, but three images
are insufficient to establish that the model has learned a specific spatial
shortcut.

The heatmaps should therefore be treated as diagnostic evidence that can
generate hypotheses rather than proof of causal model behavior.

##### 14. Confidence versus explanation

The three false negatives have different prediction confidence levels.

However, stronger heatmap intensity should not be interpreted as higher
model confidence.

These quantities answer different questions:

    Softmax probability
        ↓
    How strongly did the model prefer one class?

                    ≠

    Grad-CAM
        ↓
    Where was positive influence for the selected
    class spatially concentrated?

Grad-CAM intensity is therefore not a probability.

##### 15. Layer3 versus Layer4 experiment

Grad-CAM depends partly on the target layer selected.

Compare:

Layer3
256 channels × 14 × 14
196 spatial locations

Layer4
512 channels × 7 × 7
49 spatial locations

Use the highest-confidence false negative:

In [0]:
filename = "cast_def_0_150.jpeg"

matching_indices = [
    index
    for index, (path, label)
    in enumerate(data.test_dataset.samples)
    if path.endswith(filename)
]

if len(matching_indices) != 1:
    raise ValueError(
        f"Expected exactly one match for "
        f"{filename}, found "
        f"{len(matching_indices)}."
    )

sample_index = matching_indices[0]

image_path, _ = (
    data.test_dataset.samples[sample_index]
)

image_tensor, actual_label = (
    data.test_dataset[sample_index]
)

image_batch = image_tensor.unsqueeze(0)

In [0]:
layer3_target = (
    model.layer3[-1].conv2
)

layer4_target = (
    model.layer4[-1].conv2
)

print("Layer3 target:")
print(layer3_target)

print("\nLayer4 target:")
print(layer4_target)

In [0]:
layer3_grad_cam = GradCAM(
    model=model,
    target_layer=layer3_target,
)

layer4_grad_cam = GradCAM(
    model=model,
    target_layer=layer4_target,
)

In [0]:
layer3_result = (
    layer3_grad_cam.generate(
        input_tensor=image_batch,
        target_class=ok_class_index,
    )
)

layer4_result = (
    layer4_grad_cam.generate(
        input_tensor=image_batch,
        target_class=ok_class_index,
    )
)

In [0]:
print(
    "Layer3 activation:",
    layer3_result.activation_shape,
)

print(
    "Layer3 gradient:",
    layer3_result.gradient_shape,
)

print(
    "Layer4 activation:",
    layer4_result.activation_shape,
)

print(
    "Layer4 gradient:",
    layer4_result.gradient_shape,
)

##### 16. Visualize Layer3 versus Layer4

In [0]:
original_image = Image.open(
    image_path
).convert("RGB")

display_image = original_image.resize(
    (224, 224)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
)

axes[0].imshow(display_image)
axes[0].set_title(
    "Original Image\n"
    "Actual: def_front | "
    "Predicted: ok_front"
)
axes[0].axis("off")


axes[1].imshow(display_image)
axes[1].imshow(
    layer3_result.heatmap.numpy(),
    cmap="jet",
    alpha=0.40,
)
axes[1].set_title(
    "Layer3 Grad-CAM\n"
    "14×14 → Explain OK"
)
axes[1].axis("off")


axes[2].imshow(display_image)
axes[2].imshow(
    layer4_result.heatmap.numpy(),
    cmap="jet",
    alpha=0.40,
)
axes[2].set_title(
    "Layer4 Grad-CAM\n"
    "7×7 → Explain OK"
)
axes[2].axis("off")


plt.suptitle(
    "Grad-CAM Spatial Resolution Comparison"
)

plt.tight_layout()
plt.show()

##### 17. Layer comparison findings

##### 🔍 XAI Finding — Target Layer Selection

The Layer3 explanation uses a 14 × 14 spatial representation and therefore
contains more spatial detail than the Layer4 explanation, which uses a
7 × 7 representation.

The Layer3 visualization appears more localized and fragmented, while the
Layer4 visualization produces broader and smoother influential regions.

However, greater spatial resolution does not automatically produce a more
correct explanation.

Layer3 is earlier in the network and contains relatively lower-level visual
representations. Layer4 is later, contains more developed features, and is
closer to the final classification decision.

For this project, Layer4 remains the primary Grad-CAM target because the
goal is to explain the classification decision rather than perform precise
defect localization.

The Layer3 experiment demonstrates an important XAI principle:

> Explanations can depend on methodological choices such as the selected
> target layer. An explanation should not be treated as an absolute
> description of what the model "thinks."

##### 18. Grad-CAM limitations

Grad-CAM can help answer:

- Which broad spatial regions positively influenced a selected class score?
- Do correct and incorrect predictions show different spatial behavior?
- Are suspicious regions repeatedly influential across failure cases?
- What hypotheses should be investigated further?

Grad-CAM does **not** establish:

- the exact physical defect location;
- that highlighted pixels are defective;
- that non-highlighted pixels are defect-free;
- causal reliance on a highlighted region;
- that a correct prediction was made for the correct physical reason;
- that a high-confidence prediction is trustworthy.

The current dataset contains image-level labels but no defect bounding boxes
or segmentation masks.

Therefore, Grad-CAM must not be interpreted as ground-truth defect
localization.

If the business requirement were to identify the exact physical defect
location, a localization task such as object detection or segmentation,
together with appropriate location annotations, would be more suitable.

##### 19. XAI → RAI connection


Grad-CAM can reveal potentially suspicious model behavior, such as strong
influence from image borders, backgrounds, lighting regions, or other
unexpected areas.

Such observations generate hypotheses.

For example:

    Grad-CAM highlights background
                ↓
    possible shortcut-learning hypothesis
                ↓
    controlled perturbation or masking experiment
                ↓
    observe prediction changes
                ↓
    stronger evidence about model behavior
                ↓
    Responsible AI reliability assessment

An explanation alone does not prove that a model relies causally on a
particular visual cue.

XAI therefore provides evidence that can support a broader Responsible AI
assessment rather than replacing that assessment.

##### 20. Key Learnings


1. Grad-CAM is a post-hoc explanation technique for convolutional image
   classifiers.

2. Activations are produced during the model's normal forward pass.

3. Gradients measure how the selected class score responds to target-layer
   activations.

4. Grad-CAM averages each channel's spatial gradient values to obtain one
   class-specific importance value per channel.

5. Each importance value weights the corresponding activation grid.

6. The weighted activation grids are combined to produce one spatial
   Grad-CAM heatmap.

7. Grad-CAM is class-specific. The same image can produce different
   explanations for `ok_front` and `def_front`.

8. The model prediction does not change when a different Grad-CAM target
   class is selected. Only the class being explained changes.

9. Stronger heatmap intensity means stronger positive influence toward the
   class being explained. It does not directly mean "defect" or "no defect."

10. Prediction confidence and Grad-CAM intensity represent different
    concepts.

11. Late ResNet18 features provide a coarse 7 × 7 spatial representation,
    which produces broad Grad-CAM regions after interpolation.

12. Earlier layers can provide greater spatial resolution, but greater
    resolution does not automatically mean a better or more correct
    explanation.

13. Grad-CAM explanations depend partly on methodological choices such as
    the selected class and target layer.

14. A correct, high-confidence prediction does not prove that the model
    relied on an appropriate visual pattern.

15. Failure-case analysis across multiple images is more informative than
    interpreting a single heatmap.

16. Grad-CAM is useful for generating hypotheses about model behavior but
    does not establish causal reliance.

17. Grad-CAM is not object detection or segmentation and should not be
    presented as precise defect localization.

##### 21. Notebook Conclusion


This notebook applied Grad-CAM to the persisted fine-tuned ResNet18 product
defect classifier.

The Grad-CAM implementation captured target-layer activation grids during
the forward pass and class-specific gradient grids during the backward pass.
Spatial gradients were averaged to calculate channel importance values,
which were then used to combine the activation grids into a class-specific
heatmap.

For the primary Layer4 target, the model produced 512 activation grids with
a spatial resolution of 7 × 7. The resulting Grad-CAM heatmaps therefore
provided broad spatial explanations rather than precise localization.

A correctly classified defect image demonstrated that high confidence alone
does not explain why a model produced its prediction. Grad-CAM added spatial
information about where positive class influence was concentrated.

Three known false-negative defect images were then analyzed. For each image,
separate Grad-CAM explanations were generated for the predicted `ok_front`
class and the `def_front` class. The explanations demonstrated that Grad-CAM
is class-specific and that the model's prediction does not change when a
different class is selected for explanation.

Across the false negatives, some repeated broad spatial tendencies were
observed, but the small number of failure cases and coarse Grad-CAM
resolution do not justify concluding that the model has learned a specific
spatial shortcut.

A Layer3-versus-Layer4 experiment further demonstrated that XAI results
depend on the selected target layer. Layer3 provided greater spatial
resolution, while Layer4 produced broader explanations from more developed
features closer to the classifier.

Most importantly, the experiments established the limits of the explanation:

- Grad-CAM identifies broad class-influential regions.
- It does not identify ground-truth defect locations.
- It does not prove causal reliance on highlighted regions.
- It does not establish model reliability or trustworthiness.

Grad-CAM should therefore be used as a diagnostic tool for investigating
model behavior and generating hypotheses that can be tested further.

These findings provide the bridge from **Explainable AI** to the next phase
of the project: **Responsible AI**, where model reliability, evaluation
integrity, failure modes, appropriate use, limitations, and human oversight
will be assessed more systematically.

In [0]:
grad_cam.close()


##### Why don't we update model weights?

This is a very important distinction.

In training:

``` text

forward
   ↓
loss
   ↓
backward
   ↓
gradients
   ↓
optimizer.step()
   ↓
WEIGHTS CHANGE

```

During Grad-CAM:

``` text

forward
   ↓
selected class score
   ↓
backward
   ↓
gradients
   ↓
generate explanation

```

STOP

There is no: optimizer.step()

Therefore:

- Grad-CAM uses backpropagation to analyze the model; it does not train the model.
- We are using gradients as an explanation signal, not as an instruction to modify parameters.

What exactly are we going to capture?

Before writing code, let's understand the two critical ingredients.

Grad-CAM needs:

Activations
Gradients
You've already encountered both concepts separately.

Activations

Remember CNN:


Image
  ↓
Conv
  ↓
ReLU
  ↓
feature maps
The values produced by a layer during the forward pass are its activations.

For ResNet18:


Image
 ↓
...
 ↓
layer4
 ↓
feature maps
 ↓
avgpool
 ↓
fc
 ↓
prediction
We want the feature maps produced by a late convolutional layer.

For example, conceptually:

layer4 output


Feature map 1
┌─────────┐
│ . . . . │
│ . █ █ . │
│ . █ . . │
└─────────┘

Feature map 2
┌─────────┐
│ █ . . . │
│ █ . . . │
│ . . . . │
└─────────┘

...

Feature map 512
Each feature map responds differently to learned visual patterns.

** What do gradients tell us here?**

This is the part that connects directly to neural-network learning.

During training you learned:


Loss
 ↓
backpropagation
 ↓
gradients
 ↓
how should weights change?
Grad-CAM also uses gradients, but for a different purpose.

We ask: How sensitive is the selected class score to the activations in these feature maps?

Suppose the model produces:

def_front score = 4.8
ok_front score = 1.2
and we're explaining:

def_front

We backpropagate information from that class score toward the target convolutional layer.

Conceptually:


def_front score
       ↓
backward
       ↓
layer4 feature maps
       ↓
gradients
Large relevant gradients indicate that particular feature-map activations are influential for that class score.

So:

ACTIVATION --> "What visual patterns were detected?"

GRADIENT --> "How important were those activations for this class score?"

Grad-CAM combines both.

Why layer4?

Our ResNet18 structure is approximately:


Input
 ↓
conv1
 ↓
layer1
 ↓
layer2
 ↓
layer3
 ↓
layer4
 ↓
avgpool
 ↓
flatten
 ↓
fc
For our Grad-CAM target, we'll eventually use the last convolutional block inside layer4.

Why? We want two properties:

High-level learned features + Spatial information

Earlier layers have excellent spatial information but relatively basic features.
Later fully connected layers have class information but no useful 2D spatial map.
A late convolutional layer provides a useful compromise.

**What is a PyTorch hook?**

Normally when you execute: output = model(image)

you get the final output:


image
 ↓
ResNet18
 ↓
logits
But Grad-CAM needs something from inside the model:


image
 ↓
conv1
 ↓
...
 ↓
layer4[-1].conv2
      │
      ├── I want this output!
      │
      ↓
...
 ↓
fc
 ↓
logits
We don't want to rewrite ResNet18's forward() function just to retrieve it.

PyTorch provides hooks for this.

Think of a hook as a listener attached to a layer:

Normal forward pass │ ↓ conv2 │ ├────────→ Hook: │ "Save what came │ out of here." ↓ rest of network

The hook observes the computation without replacing the normal model architecture.

**Forward hook**

For Grad-CAM, the forward hook captures the activation produced by our target layer.

Conceptually:

``` text

Image
 ↓
forward pass
 ↓
target conv2
 ↓
activation
[B,512,7,7]
 ↓
SAVE IT
 ↓
continue normal forward pass

```

The PyTorch mechanism looks like: target_layer.register_forward_hook(function)

The function is called automatically whenever that layer executes.

For example:

def save_activation(
    module,
    inputs,
    output,
):
    activation = output

You don't manually call:

save_activation(...)

PyTorch calls it when the target layer participates in the forward pass.

That's the key idea.

**Backward hook**

Grad-CAM also needs gradients associated with that target layer.

After the forward pass we have: logits

[def_front score, ok_front score]

Suppose we're explaining: ok_front

We select that class score and perform backward propagation.

Conceptually:

``` text

ok_front score
      ↓
backward()
      ↓
...
      ↓
target conv2
      │
      └────→ backward hook
             saves gradients

```
So we now have:

``` text

Forward hook
     ↓
activations

Backward hook
     ↓
gradients

```

Those are the two ingredients Grad-CAM needs.

**Activations vs gradients**

This distinction is worth making very clear.

Suppose our target layer produces: 512 feature maps

**Activations**

Tell us: What patterns did these feature maps respond to for this image?

``` text

Image
 ↓
CNN
 ↓
Feature map activations

```

**Gradients**

Tell us roughly: How does the selected class score respond to changes in those activations?

``` text

Selected class score
        ↓
backpropagation
        ↓
gradients

```

Then Grad-CAM combines:

WHAT WAS ACTIVATED
       +
WHAT MATTERED FOR THIS CLASS
       ↓
class-specific heatmap

That's why Grad-CAM is class-specific.

The same image can potentially produce different explanations for:

def_front

versus:

ok_front

because we're asking about a different output score.

**How Grad-CAM combines them**

Suppose our activations have: [1, 512, 7, 7]

and gradients also correspond to those 512 spatial feature maps.

Grad-CAM summarizes each feature map's gradient spatially.

Conceptually:

``` text

Gradient map for feature 1
7 × 7
 ↓
average
 ↓
weight 1

Gradient map for feature 2
7 × 7
 ↓
average
 ↓
weight 2

...

Gradient map for feature 512
7 × 7
 ↓
average
 ↓
weight 512
```

We therefore get:

``` text

512 feature maps
        +
512 importance weights
        ↓
weighted combination
        ↓
one 7 × 7 map

```
Conceptually:

``` text

weight1 × feature_map1
+
weight2 × feature_map2
+
...
+
weight512 × feature_map512
        ↓
Grad-CAM
[7,7]

```

Then commonly:

Grad-CAM
  ↓
ReLU
  ↓
normalize
  ↓
resize 7×7 → image size
  ↓
overlay on image
Why ReLU?

For the standard Grad-CAM visualization, we're primarily interested in regions with positive influence toward the class being explained.

So negative values are suppressed:

negative → 0
positive → keep

We'll implement that explicitly.


``` text

                NORMAL RESNET FORWARD PASS
                ==========================

Image
[3,224,224]
      ↓
ResNet layers
      ↓
Target convolution
      ↓
512 ACTIVATION GRIDS
each [7,7]
      │
      │
      └───────────────────────────┐
                                  │
ResNet continues                  │ save these
      ↓                           │
Defect logit                      │
                                  │
                                  │
                GRAD-CAM BACKWARD
                =================
                                  │
Defect logit                      │
      ↓ backward                  │
512 GRADIENT GRIDS                │
each [7,7]                        │
      ↓                           │
average each grid                 │
      ↓                           │
512 IMPORTANCE NUMBERS            │
      │                           │
      └──────────────┬────────────┘
                     ↓
              COMBINE THEM
              ============

importance₁ × activation grid₁
+
importance₂ × activation grid₂
+
...
+
importance₅₁₂ × activation grid₅₁₂

                     ↓

             ONE [7,7] grid

                     ↓
                   ReLU
                     ↓
                 normalize
                     ↓
              resize to 224×224
                     ↓
             overlay on image

```

Forward pass
→ capture activation grids

Selected class logit
→ backward pass
→ capture gradient grids

Average gradient grids
→ channel importance numbers

Importance × activation grids
→ combine
→ Grad-CAM heatmap

The important parameter is: alpha=0.40

which makes the heatmap partially transparent.

Conceptually:

``` text

Original casting image
        +
semi-transparent Grad-CAM heatmap
        ↓
we can see which image regions
influenced the class score

```